# 13-3. 아티팩트 파서와 분석 엔진 연계 예제

## Goal

- 아티팩트별 처리 도구를 명시적으로 선택합니다.
- 명령행 원문과 실행 인자를 분리합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

외부 도구를 실행하지 않고 허용된 명령 계획만 구성합니다.


## Steps

### 파서 작업 계획 검증

실행 파일과 옵션을 고정된 목록에서 선택하고 셸 문자열을 만들지 않습니다.


In [1]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ToolSpec:
    artifact: str
    executable: Path
    output_format: str


TOOLS = {
    "evtx": ToolSpec("evtx", Path("/opt/training/EvtxECmd"), "csv"),
    "prefetch": ToolSpec("prefetch", Path("/opt/training/PECmd"), "csv"),
}


def build_command(artifact: str, input_path: Path, output_dir: Path):
    if artifact not in TOOLS:
        raise ValueError("허용하지 않은 아티팩트입니다")
    spec = TOOLS[artifact]
    if not input_path.is_absolute() or not output_dir.is_absolute():
        raise ValueError("검증된 절대 경로가 필요합니다")
    return [str(spec.executable), "--input", str(input_path), "--output", str(output_dir), "--format", spec.output_format]


command = build_command("evtx", Path("/cases/demo/evidence.evtx"), Path("/cases/demo/parser-output"))
print(command)


['/opt/training/EvtxECmd', '--input', '/cases/demo/evidence.evtx', '--output', '/cases/demo/parser-output', '--format', 'csv']


## Checks

인자 목록이 유지되고 지원하지 않는 자료는 거부되는지 확인합니다.


In [2]:
assert isinstance(command, list) and "--input" in command
assert not any(";" in part for part in command)
try:
    build_command("memory", Path("/tmp/a"), Path("/tmp/b"))
except ValueError:
    print("허용 목록 밖 도구 거부 확인")


허용 목록 밖 도구 거부 확인


## Next Steps

실제 실행 전에는 도구 버전·해시·옵션·종료 코드·출력 헤더를 함께 검증합니다.
